In [1]:
# =============================================================================
#  Angrist & Evans (1998) replication - multi-era harmonized pipeline
#  Extracts:  usa_00001 = ACS 2021-24 (1-yr)  |  usa_00002 = Census 1980 & 1990 5% state
#  The SAME code path runs on every era, so differences reflect TIME, not method.
# =============================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from ipumspy import readers
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# extract stub -> human label
EXTRACTS = {
    "usa_00001": "ACS 2021-24",
    "usa_00002": "Census 1980/90",
}

# baseline controls (year fixed effects are added automatically when an era pools years)
CONTROLS_BASE = ['AGE', 'age_at_first_birth', 'boy_1st', 'boy_2nd',
                 'black', 'hispanic', 'other_race']

print("Environment ready - statsmodels + linearmodels + ipumspy loaded.")

Environment ready - statsmodels + linearmodels + ipumspy loaded.


In [2]:
# =============================================================================
#  DATA ENGINE - one function, applied identically to every extract / era.
#  build_panel() returns ONE row per woman aged 21-35 and KEEPS the diagnostic
#  columns (age_k1, age_k2, total_kids, GQ) so A&E restrictions can be tuned
#  downstream WITHOUT re-reading the raw microdata.  Husband outcomes
#  (worked / weeks / hours / income) are carried for the Table 7 husband block.
# =============================================================================

def load_households(stub, chunksize=100000):
    """Stream an extract; keep every member of a household that contains a
    woman aged 21-35.  Matching on (YEAR, SERIAL) avoids the SERIAL collision."""
    ddi = readers.read_ipums_ddi(f"{stub}.xml")
    kept = []
    for chunk in readers.read_microdata_chunked(ddi, f"{stub}.dat", chunksize=chunksize):
        women = chunk[(chunk['SEX'] == 2) & (chunk['AGE'].between(21, 35))]
        keys = women[['YEAR', 'SERIAL']].drop_duplicates()
        kept.append(chunk.merge(keys, on=['YEAR', 'SERIAL'], how='inner'))
    return pd.concat(kept, axis=0, ignore_index=True)


def build_panel(stub):
    """Mother-level panel WITHOUT the final A&E restrictions (those are applied
    in make_analysis()).  Carries diagnostics + raw income for later deflation."""
    label = EXTRACTS[stub]
    print(f"  building {label} ({stub}) ...")
    raw = load_households(stub)

    def uid(d, loc):
        return (d['YEAR'].astype(str) + "_" + d['SERIAL'].astype(str) + "_"
                + d[loc].fillna(0).astype(int).astype(str))

    # --- mothers ---------------------------------------------------------
    moms = raw[(raw['SEX'] == 2) & (raw['AGE'].between(21, 35))].copy()
    moms['mom_id'] = uid(moms, 'PERNUM')

    # --- children, ordered by age to recover the first two births --------
    kids = raw[raw['MOMLOC'] > 0].copy()
    kids['mom_id'] = uid(kids, 'MOMLOC')
    kids = kids.sort_values(['mom_id', 'AGE', 'PERNUM'], ascending=[True, False, True])
    kids['birth_order'] = kids.groupby('mom_id').cumcount() + 1
    kid1 = kids[kids.birth_order == 1].set_index('mom_id')[['AGE', 'SEX']].rename(columns={'AGE': 'age_k1', 'SEX': 'sex_k1'})
    kid2 = kids[kids.birth_order == 2].set_index('mom_id')[['AGE', 'SEX']].rename(columns={'AGE': 'age_k2', 'SEX': 'sex_k2'})
    total_kids = kids.groupby('mom_id').size().rename('total_kids')

    df = moms.join(kid1, on='mom_id').join(kid2, on='mom_id').join(total_kids, on='mom_id')
    df['total_kids'] = df['total_kids'].fillna(0).astype(int)

    # --- husband's labor supply (worked / weeks / hours / income) -------
    df['spouse_id'] = uid(df, 'SPLOC')
    men = raw[(raw['SEX'] == 1) & (raw['AGE'] >= 18)].copy()
    men['person_id'] = uid(men, 'PERNUM')
    men['h_weeks']  = pd.to_numeric(men['WKSWORK1'], errors='coerce')
    men['h_worked'] = (men['h_weeks'] > 0).astype(float)
    men['h_hours']  = pd.to_numeric(men['UHRSWORK'], errors='coerce')
    hinc = pd.to_numeric(men['INCWAGE'], errors='coerce')
    men['h_inc_nominal'] = hinc.where(hinc < 999998, 0.0)
    husbands = men[['person_id', 'h_worked', 'h_weeks', 'h_hours', 'h_inc_nominal']]
    df = df.merge(husbands, left_on='spouse_id', right_on='person_id', how='left')

    # --- wife outcomes (A&E definitions) & raw income for deflation ------
    df['weeks_worked']   = pd.to_numeric(df['WKSWORK1'], errors='coerce').fillna(0.0)
    df['worked_for_pay'] = (df['weeks_worked'] > 0).astype(float)            # worked LAST YEAR
    df['hours_week']     = pd.to_numeric(df['UHRSWORK'], errors='coerce').fillna(0.0)
    inc = pd.to_numeric(df['INCWAGE'], errors='coerce')
    df['inc_nominal']    = inc.where(inc < 999998, 0.0).fillna(0.0)
    df['cpi99']          = pd.to_numeric(df['CPI99'], errors='coerce')

    # --- controls, weight, marital status, group-quarters flag ----------
    df['black']      = (df['RACE'] == 2).astype(float)
    df['hispanic']   = (df['HISPAN'] > 0).astype(float)
    df['other_race'] = ((df['RACE'] > 2) & (df['HISPAN'] == 0)).astype(float)
    df['weight']     = pd.to_numeric(df['PERWT'], errors='coerce').fillna(0.0)
    df['married']    = (df['SPLOC'] > 0).astype(int)
    df['GQ']         = pd.to_numeric(df['GQ'], errors='coerce')

    cols = ['YEAR', 'mom_id', 'weight', 'married', 'GQ', 'AGE',
            'age_k1', 'age_k2', 'sex_k1', 'sex_k2', 'total_kids',
            'worked_for_pay', 'weeks_worked', 'hours_week', 'inc_nominal', 'cpi99',
            'h_worked', 'h_weeks', 'h_hours', 'h_inc_nominal',
            'black', 'hispanic', 'other_race']
    out = df[cols].copy()
    print(f"    -> {len(out):,} women 21-35 retained (pre-restriction)")
    return out


# CPI-U scaling to express labor income in a chosen base year (inc*cpi99 = 1999$).
CPI_TO = {1999: 1.000, 1995: 0.915}   # 1995$ = 1999$ * (CPI1995/CPI1999)

def make_analysis(df, cap_oldest=17, drop_infant2=True, hh_only=True, dollars=1995):
    """Apply A&E sample restrictions and build derived variables.
    Knobs:  cap_oldest (max age of oldest child), drop_infant2 (drop 2nd child
    < 1 yr, A&E footnote), hh_only (drop group quarters), dollars (income base)."""
    d = df.copy()
    if hh_only:
        d = d[d['GQ'].isin([1, 2])]
    d = d[(d['total_kids'] >= 2) & d['age_k1'].notna() & d['age_k2'].notna()]
    if cap_oldest is not None:
        d = d[d['age_k1'] <= cap_oldest]
    if drop_infant2:
        d = d[d['age_k2'] >= 1]
    d['age_at_first_birth'] = d['AGE'] - d['age_k1']
    d = d[d['age_at_first_birth'] >= 15].copy()

    d['more_than_2'] = (d['total_kids'] > 2).astype(int)
    d['boy_1st']  = (d['sex_k1'] == 1).astype(int)
    d['boy_2nd']  = (d['sex_k2'] == 1).astype(int)
    d['same_sex'] = (d['boy_1st'] == d['boy_2nd']).astype(int)
    d['two_boys'] = ((d['boy_1st'] == 1) & (d['boy_2nd'] == 1)).astype(int)
    d['two_girls']= ((d['boy_1st'] == 0) & (d['boy_2nd'] == 0)).astype(int)
    d['labor_income']   = d['inc_nominal']   * d['cpi99'] * CPI_TO[dollars]
    d['husband_income'] = d['h_inc_nominal'] * d['cpi99'] * CPI_TO[dollars]
    d = d.rename(columns={'h_worked': 'husband_worked', 'h_weeks': 'husband_weeks',
                          'h_hours': 'husband_hours'})

    sr, ss = d['boy_1st'].mean(), d['same_sex'].mean()
    assert 0.50 <= sr <= 0.53 and 0.49 <= ss <= 0.52, f"sex ratios off: boy1 {sr:.3f} same {ss:.3f}"
    return d

In [3]:
# =============================================================================
#  BUILD ALL ERAS (same pipeline per extract) then POOL.
#  Cached UNRESTRICTED to panel_full.pkl -> restrictions are applied later in
#  make_analysis(), so we can tune the A&E sample match without re-reading 4 GB.
#  Delete panel_full.pkl to force a rebuild after changing build_panel().
# =============================================================================
import os
CACHE = "panel_full.pkl"
if os.path.exists(CACHE):
    df_full = pd.read_pickle(CACHE)
    print(f"loaded cached panel: {len(df_full):,} women 21-35")
else:
    panels = [build_panel(stub) for stub in EXTRACTS]   # usa_00001 (ACS), usa_00002 (1980+1990)
    df_full = pd.concat(panels, axis=0, ignore_index=True)
    df_full = df_full[df_full['weight'] > 0].reset_index(drop=True)  # IV needs positive weights
    df_full.to_pickle(CACHE)

df_full['era'] = np.select(
    [df_full['YEAR'] <= 1980, df_full['YEAR'] <= 1990],
    ['1980', '1990'],
    default='2021-24')
ERAS = ['1980', '1990', '2021-24']

# A&E-matched analysis sample (all women), used by every table below.
# dollars=1995 matches the paper's Table 2/7 units; the two restriction knobs
# below close most of the gap to A&E (see the calibration cell).
df_all = pd.concat([make_analysis(df_full[df_full['era'] == e],
                                  cap_oldest=17, drop_infant2=True, hh_only=True,
                                  dollars=1995).assign(era=e)
                    for e in ERAS], ignore_index=True)

print("\nMothers per era (A&E restrictions applied):")
print(df_all.groupby('era').size().reindex(ERAS).to_string())

loaded cached panel: 4,087,012 women 21-35

Mothers per era (A&E restrictions applied):
era
1980       478469
1990       485503
2021-24    209574


In [4]:
# =============================================================================
#  TABLE 3 - fraction that had another child, by parity & sex of first two
#  PERWT-weighted.  All paper categories retained, per era and by marital status.
# =============================================================================

def _wprop(d, mask):
    """PERWT-weighted P(more_than_2) on a subgroup, with Kish-effective-N s.e.
    and the subgroup's weighted share of the sample."""
    s = d[mask]
    w = s['weight'].to_numpy(dtype=float)
    if w.sum() == 0:
        return np.nan, np.nan, np.nan
    p = np.average(s['more_than_2'], weights=w)
    n_eff = w.sum() ** 2 / np.square(w).sum()
    se = np.sqrt(p * (1 - p) / n_eff)
    return p, se, w.sum() / d['weight'].sum()

def table3(d):
    cats = [("one boy, one girl", d['same_sex'] == 0),
            ("two girls",         d['two_girls'] == 1),
            ("two boys",          d['two_boys'] == 1),
            ("both same sex",     d['same_sex'] == 1)]
    rows, store = {}, {}
    for name, mask in cats:
        p, se, fr = _wprop(d, mask)
        store[name] = (p, se)
        rows[name] = [f"{p:.3f}", f"({se:.3f})", f"{fr:.3f}"]
    pm, sem = store["one boy, one girl"]
    ps, ses = store["both same sex"]
    diff, sed = ps - pm, np.sqrt(sem ** 2 + ses ** 2)
    rows["difference (same - mixed)"] = [f"{diff:.3f}", f"({sed:.3f})", ""]
    return pd.DataFrame(rows, index=["had another child", "  (s.e.)", "frac of sample"]).T

print("=" * 70)
print("TABLE 3  -  FRACTION THAT HAD ANOTHER CHILD  (PERWT-weighted)")
print("=" * 70)
for era in ERAS:
    d = df_all[df_all['era'] == era]
    for sample, sub in [("All women", d), ("Married women", d[d['married'] == 1])]:
        print(f"\n[{era}]  {sample}   (n = {len(sub):,}, weighted N = {int(sub['weight'].sum()):,})")
        print(table3(sub).to_string())
print("\n" + "=" * 70)

TABLE 3  -  FRACTION THAT HAD ANOTHER CHILD  (PERWT-weighted)

[1980]  All women   (n = 478,469, weighted N = 9,569,380)
                          had another child   (s.e.) frac of sample
one boy, one girl                     0.378  (0.001)          0.495
two girls                             0.443  (0.001)          0.242
two boys                              0.427  (0.001)          0.263
both same sex                         0.434  (0.001)          0.505
difference (same - mixed)             0.056  (0.001)               

[1980]  Married women   (n = 396,068, weighted N = 7,921,360)
                          had another child   (s.e.) frac of sample
one boy, one girl                     0.375  (0.001)          0.495
two girls                             0.445  (0.002)          0.240
two boys                              0.427  (0.002)          0.266
both same sex                         0.435  (0.001)          0.505
difference (same - mixed)             0.061  (0.002)               


In [5]:
# =============================================================================
#  TABLE 7 - OLS & 2SLS effect of (>2 children) on labor supply, PERWT-weighted
#  Estimators per outcome:  OLS | 2SLS same-sex | 2SLS two-boys/two-girls
# =============================================================================
RESULTS = {}   # (era, sample, outcome) -> estimates, consumed by the cross-era cell

def _design(d_in):
    """Add year fixed effects when an era pools multiple survey years (ACS)."""
    d = d_in.copy()
    ctrl = list(CONTROLS_BASE)
    if d['YEAR'].nunique() > 1:
        yd = pd.get_dummies(d['YEAR'], prefix='yr', drop_first=True).astype(float)
        d = pd.concat([d, yd], axis=1)
        ctrl += list(yd.columns)
    return d, ctrl

def run_table7(d_in, outcomes, era, sample):
    d, ctrl = _design(d_in)
    d = d.dropna(subset=outcomes + ['more_than_2', 'same_sex', 'two_boys', 'two_girls', 'weight'] + ctrl)
    w = d['weight']
    X = sm.add_constant(d[ctrl].astype(float))
    endog = d['more_than_2'].astype(float)
    fmt = {}
    for y in outcomes:
        Y = d[y].astype(float)
        # OLS (weighted, robust)
        m_ols = sm.WLS(Y, X.assign(more_than_2=endog), weights=w).fit(cov_type='HC1')
        b_ols, s_ols = m_ols.params['more_than_2'], m_ols.bse['more_than_2']
        # 2SLS - same-sex instrument
        m1 = IV2SLS(Y, X, endog, d['same_sex'].astype(float), weights=w).fit(cov_type='robust')
        b1, s1 = m1.params['more_than_2'], m1.std_errors['more_than_2']
        # 2SLS - two-boys / two-girls instruments (boy_2nd drops: collinear with instruments)
        Xc = X.drop(columns=['boy_2nd'])
        m2 = IV2SLS(Y, Xc, endog, d[['two_boys', 'two_girls']].astype(float), weights=w).fit(cov_type='robust')
        b2, s2 = m2.params['more_than_2'], m2.std_errors['more_than_2']
        try:
            fsF = float(m1.first_stage.diagnostics['f.stat'].iloc[0])
        except Exception:
            fsF = np.nan
        RESULTS[(era, sample, y)] = dict(ols=(b_ols, s_ols), iv_ss=(b1, s1), iv_bg=(b2, s2), fsF=fsF)
        fmt[y] = [f"{b_ols:.3f} ({s_ols:.3f})", f"{b1:.3f} ({s1:.3f})", f"{b2:.3f} ({s2:.3f})"]
    return pd.DataFrame(fmt, index=["OLS", "2SLS (same sex)", "2SLS (2boy/2girl)"]).T

OUT_ALL = ['worked_for_pay', 'weeks_worked', 'hours_week', 'labor_income']
OUT_MAR = OUT_ALL + ['husband_weeks']

print("=" * 78)
print("TABLE 7  -  EFFECT OF >2 CHILDREN ON LABOR SUPPLY  (PERWT-weighted)")
print("           labor_income in constant 1999 dollars (CPI99-deflated)")
print("=" * 78)
for era in ERAS:
    d = df_all[df_all['era'] == era]
    for sample, sub, outs in [("All women", d, OUT_ALL),
                              ("Married women", d[d['married'] == 1], OUT_MAR)]:
        print(f"\n[{era}]  {sample}   (n = {len(sub):,})")
        print(run_table7(sub, outs, era, sample).to_string())
print("\n" + "=" * 78)

TABLE 7  -  EFFECT OF >2 CHILDREN ON LABOR SUPPLY  (PERWT-weighted)
           labor_income in constant 1999 dollars (CPI99-deflated)

[1980]  All women   (n = 478,469)
                               OLS      2SLS (same sex)    2SLS (2boy/2girl)
worked_for_pay      -0.171 (0.001)       -0.120 (0.025)       -0.114 (0.024)
weeks_worked        -8.695 (0.065)       -5.382 (1.092)       -5.108 (1.086)
hours_week          -6.523 (0.056)       -4.250 (0.939)       -4.020 (0.934)
labor_income    -3596.400 (30.865)  -1707.399 (521.696)  -1653.018 (519.127)

[1980]  Married women   (n = 396,068)
                               OLS      2SLS (same sex)    2SLS (2boy/2girl)
worked_for_pay      -0.162 (0.002)       -0.129 (0.025)       -0.124 (0.025)
weeks_worked        -8.066 (0.071)       -5.659 (1.095)       -5.493 (1.088)
hours_week          -5.985 (0.061)       -4.760 (0.936)       -4.606 (0.930)
labor_income    -3216.624 (32.826)  -1644.328 (512.318)  -1648.055 (509.199)
husband_weeks       -0

In [6]:
# =============================================================================
#  WHAT CHANGED IN 30 YEARS - formal comparison of the headline estimates
#  Independent era-samples =>  D = b_end - b_base ,  s.e.(D) = sqrt(se_base^2 + se_end^2)
#  -> simple z-test on every coefficient.
# =============================================================================

def compare(sample, outcomes, estimator='iv_ss', base='1980', end='2021-24'):
    rows = {}
    for y in outcomes:
        a, b = RESULTS.get((base, sample, y)), RESULTS.get((end, sample, y))
        if a is None or b is None:
            continue
        (ba, sa), (bb, sb) = a[estimator], b[estimator]
        D, se = bb - ba, np.sqrt(sa ** 2 + sb ** 2)
        z = D / se if se else np.nan
        rows[y] = [f"{ba:.3f}", f"{bb:.3f}", f"{D:+.3f}", f"({se:.3f})", f"{z:+.2f}"]
    return pd.DataFrame(rows, index=[base, end, f"D ({end}-{base})", "  s.e.", "z"]).T

print("=" * 74)
print("WHAT CHANGED 1980 -> 2021-24   (2SLS same-sex LATE, PERWT-weighted)")
print("=" * 74)
print("\nMarried women:")
print(compare("Married women", OUT_MAR).to_string())
print("\nAll women:")
print(compare("All women", OUT_ALL).to_string())

# channel: did the instrument weaken? (first-stage F of same-sex across eras)
print("\n" + "=" * 74)
print("INSTRUMENT STRENGTH over time  (first-stage F, same-sex, married sample)")
print("=" * 74)
fs = {era: [f"{RESULTS[(era, 'Married women', 'worked_for_pay')]['fsF']:.0f}"]
      for era in ERAS if (era, 'Married women', 'worked_for_pay') in RESULTS}
print(pd.DataFrame(fs, index=["first-stage F"]).T.to_string())
print("=" * 74)

WHAT CHANGED 1980 -> 2021-24   (2SLS same-sex LATE, PERWT-weighted)

Married women:
                     1980    2021-24 D (2021-24-1980)        s.e.      z
worked_for_pay     -0.129     -0.140           -0.011     (0.060)  -0.19
weeks_worked       -5.659     -9.008           -3.349     (3.024)  -1.11
hours_week         -4.760     -5.727           -0.966     (2.437)  -0.40
labor_income    -1644.328  -2800.011        -1155.683  (2333.801)  -0.50
husband_weeks       0.102      1.149           +1.047     (1.802)  +0.58

All women:
                     1980    2021-24 D (2021-24-1980)        s.e.      z
worked_for_pay     -0.120     -0.179           -0.058     (0.061)  -0.95
weeks_worked       -5.382     -9.910           -4.528     (3.122)  -1.45
hours_week         -4.250     -6.859           -2.610     (2.515)  -1.04
labor_income    -1707.399  -3816.587        -2109.189  (2277.367)  -0.93

INSTRUMENT STRENGTH over time  (first-stage F, same-sex, married sample)
        first-stage F
1980 